In [1]:
!pip -q install pandas pyarrow numpy tqdm sentence-transformers faiss-cpu rank-bm25 scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 54.9 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from google.colab import drive

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PROJECT_DIR = "/content/drive/MyDrive/finance-rag-analyst"

PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
INDEX_DIR = f"{PROJECT_DIR}/data/indexes"
EVAL_DIR = f"{PROJECT_DIR}/data/evaluation"

os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

CHUNKS_PATH = f"{PROCESSED_DIR}/sec_filing_chunks.parquet"

EMBEDDINGS_PATH = f"{INDEX_DIR}/bge_small_embeddings.npy"
FAISS_INDEX_PATH = f"{INDEX_DIR}/faiss_bge_small.index"
BM25_INDEX_PATH = f"{INDEX_DIR}/bm25_index.pkl"
CHUNK_METADATA_PATH = f"{INDEX_DIR}/chunk_metadata.parquet"

EVAL_RESULTS_PATH = f"{EVAL_DIR}/retrieval_eval_results.csv"
BEST_CONFIG_PATH = f"{EVAL_DIR}/best_retrieval_config.json"

print("Chunks path:", CHUNKS_PATH)
print("Index dir:", INDEX_DIR)
print("Eval dir:", EVAL_DIR)

Chunks path: /content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filing_chunks.parquet
Index dir: /content/drive/MyDrive/finance-rag-analyst/data/indexes
Eval dir: /content/drive/MyDrive/finance-rag-analyst/data/evaluation


Load chunks

In [5]:
assert os.path.exists(CHUNKS_PATH), f"Missing chunks file: {CHUNKS_PATH}"

chunks_df = pd.read_parquet(CHUNKS_PATH)

print("Chunks loaded:", len(chunks_df))
display(chunks_df.head())

Chunks loaded: 2093


,chunk_id,document_id,ticker,company,cik,form_type,filing_date,report_date,accession_number,primary_document,...,chunk_char_start,chunk_char_end,text,char_count,word_count,topic_labels,document_char_count,chunk_position_percent,estimated_tokens,text_hash
0,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,0001045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021,nvda-20260125.htm,...,0,3970,nvda-20260125 Table of Contents UNITED STATES ...,3970,648,regulation,341338,0.0000,842,0b0c8c295775c6f1
1,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,0001045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021,nvda-20260125.htm,...,3370,7228,required a recovery analysis of incentive-base...,3858,584,advertising|cybersecurity|mda|regulation|risk_...,341338,0.0099,759,c3c00796240da379
2,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,0001045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021,nvda-20260125.htm,...,6628,10508,dance at upcoming investor and industry confer...,3880,574,advertising|ai_strategy|data_centers|regulatio...,341338,0.0194,746,fbb7a35fbe6befa9
3,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,0001045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021,nvda-20260125.htm,...,9908,13722,"mation may be limited or incomplete, and our s...",3814,550,advertising|ai_strategy|business|data_centers|...,341338,0.0290,715,086412e9fe5635c7
4,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,0001045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021,nvda-20260125.htm,...,13122,17064,rkets with the same underlying technology by u...,3942,598,acquisitions|advertising|ai_strategy|business|...,341338,0.0384,777,c5c768e38506ab18


Validate chunk dataset

In [6]:
required_cols = [
    "chunk_id",
    "document_id",
    "ticker",
    "company",
    "form_type",
    "filing_date",
    "report_date",
    "accession_number",
    "source_url",
    "chunk_index",
    "text",
    "char_count",
    "word_count",
    "topic_labels",
    "estimated_tokens",
]

missing = [col for col in required_cols if col not in chunks_df.columns]
assert not missing, f"Missing columns: {missing}"

assert len(chunks_df) > 500, f"Too few chunks: {len(chunks_df)}"
assert chunks_df["chunk_id"].is_unique, "chunk_id is not unique."
assert chunks_df["text"].notna().all(), "Null text found."
assert chunks_df["source_url"].notna().all(), "Null source_url found."

chunks_df = chunks_df.reset_index(drop=True)
chunks_df["row_id"] = chunks_df.index

chunks_df["ticker"] = chunks_df["ticker"].astype(str).str.upper().str.strip()
chunks_df["text"] = chunks_df["text"].astype(str)
chunks_df["topic_labels"] = chunks_df["topic_labels"].fillna("general").astype(str)

print("Validation passed.")
print("Tickers:", sorted(chunks_df["ticker"].unique()))
print("Forms:", chunks_df["form_type"].value_counts().to_dict())
print("Median chunk chars:", int(chunks_df["char_count"].median()))
print("Median estimated tokens:", int(chunks_df["estimated_tokens"].median()))

Validation passed.
Tickers: ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'NVDA']
Forms: {'10-K': 1401, '10-Q': 692}
Median chunk chars: 3903
Median estimated tokens: 776


Save chunk metadata

In [7]:
metadata_cols = [
    "row_id",
    "chunk_id",
    "document_id",
    "ticker",
    "company",
    "form_type",
    "filing_date",
    "report_date",
    "accession_number",
    "source_url",
    "chunk_index",
    "chunk_char_start",
    "chunk_char_end",
    "char_count",
    "word_count",
    "estimated_tokens",
    "topic_labels",
    "text",
]

available_metadata_cols = [c for c in metadata_cols if c in chunks_df.columns]
chunk_metadata_df = chunks_df[available_metadata_cols].copy()

chunk_metadata_df.to_parquet(CHUNK_METADATA_PATH, index=False)

print("Saved metadata:", CHUNK_METADATA_PATH)
print("Rows:", len(chunk_metadata_df))

Saved metadata: /content/drive/MyDrive/finance-rag-analyst/data/indexes/chunk_metadata.parquet
Rows: 2093


Load embedding model

In [8]:
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Loaded embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


/tmp/ipykernel_474/102602143.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


Build or load embeddings

In [9]:
texts = chunks_df["text"].tolist()

if os.path.exists(EMBEDDINGS_PATH):
    embeddings = np.load(EMBEDDINGS_PATH)
    print("Loaded existing embeddings:", EMBEDDINGS_PATH)
else:
    print("Creating embeddings...")
    embeddings = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    np.save(EMBEDDINGS_PATH, embeddings)
    print("Saved embeddings:", EMBEDDINGS_PATH)

print("Embeddings shape:", embeddings.shape)

assert embeddings.shape[0] == len(chunks_df), "Embedding row count mismatch."
assert embeddings.dtype == np.float32, "Embeddings must be float32 for FAISS."

Loaded existing embeddings: /content/drive/MyDrive/finance-rag-analyst/data/indexes/bge_small_embeddings.npy
Embeddings shape: (2093, 384)


Build or load FAISS index

In [10]:
embedding_dim = embeddings.shape[1]

if os.path.exists(FAISS_INDEX_PATH):
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    print("Loaded FAISS index:", FAISS_INDEX_PATH)
else:
    faiss_index = faiss.IndexFlatIP(embedding_dim)
    faiss_index.add(embeddings)
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    print("Saved FAISS index:", FAISS_INDEX_PATH)

print("FAISS vectors:", faiss_index.ntotal)
assert faiss_index.ntotal == len(chunks_df), "FAISS index count mismatch."

Loaded FAISS index: /content/drive/MyDrive/finance-rag-analyst/data/indexes/faiss_bge_small.index
FAISS vectors: 2093


Build or load BM25 index

In [11]:
def tokenize_for_bm25(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9$%.\-]+", " ", text)
    tokens = text.split()
    return [tok for tok in tokens if len(tok) > 1]


if os.path.exists(BM25_INDEX_PATH):
    with open(BM25_INDEX_PATH, "rb") as f:
        bm25_payload = pickle.load(f)

    bm25_index = bm25_payload["bm25_index"]
    tokenized_corpus = bm25_payload["tokenized_corpus"]
    print("Loaded BM25 index:", BM25_INDEX_PATH)
else:
    print("Building BM25 index...")
    tokenized_corpus = [tokenize_for_bm25(text) for text in tqdm(texts)]
    bm25_index = BM25Okapi(tokenized_corpus)

    bm25_payload = {
        "bm25_index": bm25_index,
        "tokenized_corpus": tokenized_corpus,
    }

    with open(BM25_INDEX_PATH, "wb") as f:
        pickle.dump(bm25_payload, f)

    print("Saved BM25 index:", BM25_INDEX_PATH)

print("BM25 corpus size:", len(tokenized_corpus))
assert len(tokenized_corpus) == len(chunks_df), "BM25 corpus count mismatch."

Loaded BM25 index: /content/drive/MyDrive/finance-rag-analyst/data/indexes/bm25_index.pkl
BM25 corpus size: 2093


Query expansion

In [12]:
QUERY_EXPANSIONS = {
    "ai": ["artificial intelligence", "generative ai", "machine learning", "accelerated computing"],
    "capex": ["capital expenditures", "capital investment", "infrastructure investment", "data centers"],
    "cloud": ["aws", "azure", "google cloud", "cloud services"],
    "chips": ["semiconductor", "gpu", "accelerator", "blackwell"],
    "china": ["greater china", "export controls", "trade restrictions"],
    "risk": ["risk factors", "could adversely affect", "uncertainty"],
    "advertising": ["ads", "ad revenue", "advertising revenue"],
}

def has_word(text, word):
    return re.search(rf"\b{re.escape(word)}\b", text.lower()) is not None

def expand_query(query, enabled=True):
    if not enabled:
        return query

    lowered = query.lower()
    expanded_terms = []

    for key, values in QUERY_EXPANSIONS.items():
        if has_word(lowered, key):
            expanded_terms.extend(values)

    if expanded_terms:
        return query + " " + " ".join(sorted(set(expanded_terms)))

    return query

Filtering utilities

In [13]:
def build_filter_mask(tickers=None, forms=None):
    mask = np.ones(len(chunks_df), dtype=bool)

    if tickers is not None:
        tickers = [t.upper().strip() for t in tickers]
        mask &= chunks_df["ticker"].isin(tickers).values

    if forms is not None:
        forms = [f.strip() for f in forms]
        mask &= chunks_df["form_type"].isin(forms).values

    return mask


def apply_mask_to_results(row_ids, scores, mask):
    filtered = []
    for row_id, score in zip(row_ids, scores):
        if row_id >= 0 and mask[row_id]:
            filtered.append((int(row_id), float(score)))
    return filtered

Company intent detection

In [14]:
COMPANY_ALIASES = {
    "NVDA": ["nvidia", "nvda"],
    "MSFT": ["microsoft", "msft", "azure"],
    "AAPL": ["apple", "aapl", "iphone", "mac"],
    "AMZN": ["amazon", "amzn", "aws"],
    "GOOGL": ["google", "alphabet", "googl", "youtube", "google cloud"],
}


def detect_tickers_from_query(query):
    lowered = query.lower()
    detected = []

    for ticker, aliases in COMPANY_ALIASES.items():
        for alias in aliases:
            pattern = rf"\b{re.escape(alias.lower())}\b"
            if re.search(pattern, lowered):
                detected.append(ticker)
                break

    return sorted(set(detected))

Dense FAISS search

In [15]:
def dense_search(query, top_k=10, tickers=None, forms=None, expand=True, search_multiplier=5):
    expanded_query = expand_query(query, enabled=expand)

    query_emb = model.encode(
        [expanded_query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    mask = build_filter_mask(tickers=tickers, forms=forms)

    if tickers is not None or forms is not None:
        candidate_k = len(chunks_df)
    else:
        candidate_k = min(len(chunks_df), max(top_k * search_multiplier, top_k))
    scores, row_ids = faiss_index.search(query_emb, candidate_k)

    filtered = apply_mask_to_results(row_ids[0], scores[0], mask)
    filtered = filtered[:top_k]

    results = []

    for rank, (row_id, score) in enumerate(filtered, start=1):
        row = chunks_df.iloc[row_id]
        results.append({
            "rank": rank,
            "row_id": row_id,
            "chunk_id": row["chunk_id"],
            "ticker": row["ticker"],
            "company": row["company"],
            "form_type": row["form_type"],
            "filing_date": row["filing_date"],
            "topic_labels": row["topic_labels"],
            "retrieval_mode": "dense",
            "retrieval_score": score,
            "source_url": row["source_url"],
            "text": row["text"],
            "text_preview": row["text"][:500]
        })

    return pd.DataFrame(results)

BM25 search

In [16]:
def bm25_search(query, top_k=10, tickers=None, forms=None, expand=True):
    expanded_query = expand_query(query, enabled=expand)
    query_tokens = tokenize_for_bm25(expanded_query)

    scores = bm25_index.get_scores(query_tokens)
    mask = build_filter_mask(tickers=tickers, forms=forms)

    valid_indices = np.where(mask)[0]

    if len(valid_indices) == 0:
        return pd.DataFrame()

    valid_scores = scores[valid_indices]
    sorted_local = np.argsort(valid_scores)[::-1][:top_k]
    top_row_ids = valid_indices[sorted_local]

    results = []

    for rank, row_id in enumerate(top_row_ids, start=1):
        row = chunks_df.iloc[row_id]
        results.append({
            "rank": rank,
            "row_id": int(row_id),
            "chunk_id": row["chunk_id"],
            "ticker": row["ticker"],
            "company": row["company"],
            "form_type": row["form_type"],
            "filing_date": row["filing_date"],
            "topic_labels": row["topic_labels"],
            "retrieval_mode": "bm25",
            "retrieval_score": float(scores[row_id]),
            "source_url": row["source_url"],
            "text": row["text"],
            "text_preview": row["text"][:500]
        })

    return pd.DataFrame(results)

Hybrid RRF search

In [17]:
def reciprocal_rank_fusion(result_dfs, rrf_k=60):
    fused_scores = {}
    source_modes = {}

    for df in result_dfs:
        if df is None or len(df) == 0:
            continue

        mode = df["retrieval_mode"].iloc[0]

        for _, row in df.iterrows():
            row_id = int(row["row_id"])
            rank = int(row["rank"])

            fused_scores[row_id] = fused_scores.get(row_id, 0.0) + 1.0 / (rrf_k + rank)
            source_modes.setdefault(row_id, []).append(mode)

    sorted_items = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)

    return sorted_items, source_modes


def diversify_results(sorted_items, final_top_k=10, max_chunks_per_document=3, max_chunks_per_ticker=5):
    selected = []
    doc_counts = {}
    ticker_counts = {}

    for row_id, score in sorted_items:
        row = chunks_df.iloc[row_id]
        doc_id = row["document_id"]
        ticker = row["ticker"]

        if doc_counts.get(doc_id, 0) >= max_chunks_per_document:
            continue

        if ticker_counts.get(ticker, 0) >= max_chunks_per_ticker:
            continue

        selected.append((row_id, score))
        doc_counts[doc_id] = doc_counts.get(doc_id, 0) + 1
        ticker_counts[ticker] = ticker_counts.get(ticker, 0) + 1

        if len(selected) >= final_top_k:
            break

    return selected


def hybrid_search(
    query,
    dense_top_k=20,
    bm25_top_k=20,
    final_top_k=10,
    rrf_k=60,
    tickers=None,
    forms=None,
    expand=True,
    diversify=True,
    max_chunks_per_document=3,
    max_chunks_per_ticker=5,
):
    dense_df = dense_search(
        query,
        top_k=dense_top_k,
        tickers=tickers,
        forms=forms,
        expand=expand
    )

    bm25_df = bm25_search(
        query,
        top_k=bm25_top_k,
        tickers=tickers,
        forms=forms,
        expand=expand
    )

    sorted_items, source_modes = reciprocal_rank_fusion([dense_df, bm25_df], rrf_k=rrf_k)

    if diversify:
        selected = diversify_results(
            sorted_items,
            final_top_k=final_top_k,
            max_chunks_per_document=max_chunks_per_document,
            max_chunks_per_ticker=max_chunks_per_ticker
        )
    else:
        selected = sorted_items[:final_top_k]

    results = []

    for rank, (row_id, score) in enumerate(selected, start=1):
        row = chunks_df.iloc[row_id]
        results.append({
            "rank": rank,
            "row_id": int(row_id),
            "chunk_id": row["chunk_id"],
            "ticker": row["ticker"],
            "company": row["company"],
            "form_type": row["form_type"],
            "filing_date": row["filing_date"],
            "topic_labels": row["topic_labels"],
            "retrieval_mode": "hybrid_rrf",
            "retrieval_score": float(score),
            "source_modes": "|".join(sorted(set(source_modes.get(row_id, [])))),
            "source_url": row["source_url"],
            "text": row["text"],
            "text_preview": row["text"][:500]
        })

    return pd.DataFrame(results)

Quick smoke tests

In [18]:
test_query = "How does Nvidia discuss demand for AI infrastructure?"

print("Dense")
display(dense_search(test_query, top_k=5)[[
    "rank", "ticker", "form_type", "filing_date", "topic_labels", "retrieval_score", "text_preview"
]])

print("BM25")
display(bm25_search(test_query, top_k=5)[[
    "rank", "ticker", "form_type", "filing_date", "topic_labels", "retrieval_score", "text_preview"
]])

print("Hybrid")
display(hybrid_search(test_query, final_top_k=5)[[
    "rank", "ticker", "form_type", "filing_date", "topic_labels", "retrieval_score", "source_modes", "text_preview"
]])

Dense


,rank,ticker,form_type,filing_date,topic_labels,retrieval_score,text_preview
0,1,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|climate_esg|c...,0.818868,"run AI models, product offerings, and services..."
1,2,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|data_centers|...,0.796559,"mation may be limited or incomplete, and our s..."
2,3,NVDA,10-K,2025-02-26,advertising|ai_strategy|business|cloud|custome...,0.782601,of the supercomputers on the global TOP500 lis...
3,4,NVDA,10-K,2024-02-21,advertising|ai_strategy|business|cloud|custome...,0.780023,of the supercomputers on the global TOP500 lis...
4,5,NVDA,10-K,2024-02-21,acquisitions|advertising|ai_strategy|business|...,0.776635,dous acceleration for applications. These plat...


BM25


,rank,ticker,form_type,filing_date,topic_labels,retrieval_score,text_preview
0,1,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|data_centers|...,35.691093,"mation may be limited or incomplete, and our s..."
1,2,NVDA,10-K,2025-02-26,advertising|ai_strategy|business|data_centers|...,33.695682,", and our statements should not be read to ind..."
2,3,NVDA,10-K,2024-02-21,advertising|ai_strategy|business|data_centers|...,33.679055,", and our statements should not be read to ind..."
3,4,NVDA,10-K,2026-02-25,ai_strategy|data_centers|privacy|semiconductor...,31.730586,or the AV market under the DRIVE Hyperion plat...
4,5,NVDA,10-K,2025-02-26,ai_strategy|cloud|data_centers|privacy|semicon...,29.997851,ying technology by using a variety of software...


Hybrid


,rank,ticker,form_type,filing_date,topic_labels,retrieval_score,source_modes,text_preview
0,1,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|data_centers|...,0.032522,bm25|dense,"mation may be limited or incomplete, and our s..."
1,2,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|climate_esg|c...,0.031319,bm25|dense,"run AI models, product offerings, and services..."
2,3,NVDA,10-K,2025-02-26,advertising|ai_strategy|business|cloud|custome...,0.030366,bm25|dense,of the supercomputers on the global TOP500 lis...
3,4,NVDA,10-K,2024-02-21,advertising|ai_strategy|business|cloud|custome...,0.030331,bm25|dense,of the supercomputers on the global TOP500 lis...
4,5,NVDA,10-K,2026-02-25,ai_strategy|data_centers|privacy|semiconductor...,0.030118,bm25|dense,or the AV market under the DRIVE Hyperion plat...


Evaluation query set

In [19]:
eval_queries = [
    {
        "query": "How does Nvidia discuss demand for AI infrastructure?",
        "expected_tickers": ["NVDA"],
        "expected_topics": ["ai_strategy", "data_centers", "semiconductors"],
        "expected_keywords": ["ai", "accelerated computing", "gpu", "data center"]
    },
    {
        "query": "What does Nvidia say about export controls and China restrictions?",
        "expected_tickers": ["NVDA"],
        "expected_topics": ["export_controls", "china", "risk_factors"],
        "expected_keywords": ["export", "china", "restrictions"]
    },
    {
        "query": "How does Microsoft discuss Azure and artificial intelligence?",
        "expected_tickers": ["MSFT"],
        "expected_topics": ["cloud", "ai_strategy"],
        "expected_keywords": ["azure", "ai", "cloud"]
    },
    {
        "query": "What does Microsoft say about cloud revenue and demand?",
        "expected_tickers": ["MSFT"],
        "expected_topics": ["cloud", "revenue", "customer_demand"],
        "expected_keywords": ["cloud", "revenue", "demand"]
    },
    {
        "query": "What risks does Apple mention related to China?",
        "expected_tickers": ["AAPL"],
        "expected_topics": ["china", "risk_factors", "supply_chain"],
        "expected_keywords": ["china", "risk", "supply"]
    },
    {
        "query": "How does Apple discuss services revenue?",
        "expected_tickers": ["AAPL"],
        "expected_topics": ["revenue", "business"],
        "expected_keywords": ["services", "revenue", "sales"]
    },
    {
        "query": "What does Amazon say about AWS and cloud infrastructure?",
        "expected_tickers": ["AMZN"],
        "expected_topics": ["cloud", "data_centers", "capital_expenditure"],
        "expected_keywords": ["aws", "cloud", "infrastructure"]
    },
    {
        "query": "How does Amazon discuss capital expenditures and data centers?",
        "expected_tickers": ["AMZN"],
        "expected_topics": ["capital_expenditure", "data_centers", "cloud"],
        "expected_keywords": ["capital expenditures", "data centers", "infrastructure"]
    },
    {
        "query": "What does Google say about advertising revenue?",
        "expected_tickers": ["GOOGL"],
        "expected_topics": ["advertising", "revenue"],
        "expected_keywords": ["advertising", "revenue", "ads"]
    },
    {
        "query": "How does Google discuss cloud services and AI?",
        "expected_tickers": ["GOOGL"],
        "expected_topics": ["cloud", "ai_strategy"],
        "expected_keywords": ["cloud", "ai", "google cloud"]
    },
    {
        "query": "Which companies discuss data center infrastructure investments?",
        "expected_tickers": ["NVDA", "MSFT", "AMZN", "GOOGL"],
        "expected_topics": ["data_centers", "capital_expenditure", "ai_strategy"],
        "expected_keywords": ["data center", "infrastructure", "capital"]
    },
    {
        "query": "Which filings discuss cybersecurity risks?",
        "expected_tickers": ["NVDA", "MSFT", "AAPL", "AMZN", "GOOGL"],
        "expected_topics": ["cybersecurity", "risk_factors"],
        "expected_keywords": ["cybersecurity", "security", "breach"]
    },
    {
        "query": "What do companies say about competition in cloud or AI markets?",
        "expected_tickers": ["MSFT", "AMZN", "GOOGL", "NVDA"],
        "expected_topics": ["competition", "cloud", "ai_strategy"],
        "expected_keywords": ["competition", "competitive", "cloud", "ai"]
    },
    {
        "query": "How do companies discuss supply chain and supplier risks?",
        "expected_tickers": ["NVDA", "AAPL", "AMZN"],
        "expected_topics": ["supply_chain", "risk_factors"],
        "expected_keywords": ["supplier", "supply chain", "inventory"]
    },
    {
        "query": "What do filings say about liquidity and cash flow?",
        "expected_tickers": ["NVDA", "MSFT", "AAPL", "AMZN", "GOOGL"],
        "expected_topics": ["liquidity", "cash_flow"],
        "expected_keywords": ["liquidity", "cash flow", "cash equivalents"]
    },
    {
        "query": "What does Apple say about share repurchases and dividends?",
        "expected_tickers": ["AAPL"],
        "expected_topics": ["share_repurchases", "dividends"],
        "expected_keywords": ["repurchase", "dividend"]
    },
    {
        "query": "What does Microsoft say about cybersecurity and security products?",
        "expected_tickers": ["MSFT"],
        "expected_topics": ["cybersecurity", "software"],
        "expected_keywords": ["security", "cybersecurity", "software"]
    },
    {
        "query": "How does Nvidia discuss Blackwell and GPUs?",
        "expected_tickers": ["NVDA"],
        "expected_topics": ["semiconductors", "ai_strategy"],
        "expected_keywords": ["blackwell", "gpu", "accelerated"]
    },
    {
        "query": "What does Amazon say about advertising?",
        "expected_tickers": ["AMZN"],
        "expected_topics": ["advertising", "revenue"],
        "expected_keywords": ["advertising", "ads"]
    },
    {
        "query": "How do filings discuss inflation and macroeconomic conditions?",
        "expected_tickers": ["NVDA", "MSFT", "AAPL", "AMZN", "GOOGL"],
        "expected_topics": ["macroeconomics", "risk_factors"],
        "expected_keywords": ["inflation", "macroeconomic", "interest rates"]
    },
]

eval_df = pd.DataFrame(eval_queries)
print("Evaluation queries:", len(eval_df))
display(eval_df.head())

Evaluation queries: 20


,query,expected_tickers,expected_topics,expected_keywords
0,How does Nvidia discuss demand for AI infrastr...,[NVDA],"[ai_strategy, data_centers, semiconductors]","[ai, accelerated computing, gpu, data center]"
1,What does Nvidia say about export controls and...,[NVDA],"[export_controls, china, risk_factors]","[export, china, restrictions]"
2,How does Microsoft discuss Azure and artificia...,[MSFT],"[cloud, ai_strategy]","[azure, ai, cloud]"
3,What does Microsoft say about cloud revenue an...,[MSFT],"[cloud, revenue, customer_demand]","[cloud, revenue, demand]"
4,What risks does Apple mention related to China?,[AAPL],"[china, risk_factors, supply_chain]","[china, risk, supply]"


Relevance scoring helpers

In [20]:
def normalize_list_field(x):
    if isinstance(x, list):
        return [str(i).lower().strip() for i in x]
    if isinstance(x, str):
        return [i.lower().strip() for i in x.split("|") if i.strip()]
    return []


def row_is_relevant(row, expected_tickers, expected_topics, expected_keywords):
    ticker = str(row["ticker"]).lower()
    topics = str(row.get("topic_labels", "")).lower()
    text = str(row.get("text", "")).lower()

    ticker_hit = ticker in expected_tickers

    topic_hit = any(topic and topic in topics for topic in expected_topics)
    keyword_hit = any(keyword and keyword.lower() in text for keyword in expected_keywords)

    # Stricter than before:
    # company-specific query: must hit ticker + keyword
    # broad query: must hit keyword + either ticker or topic
    if len(expected_tickers) == 1:
        relevant = ticker_hit and keyword_hit
    else:
        relevant = keyword_hit and (ticker_hit or topic_hit)

    return relevant, ticker_hit, topic_hit, keyword_hit


def evaluate_single_query(results_df, expected_tickers, expected_topics, expected_keywords, k=10):
    expected_tickers = normalize_list_field(expected_tickers)
    expected_topics = normalize_list_field(expected_topics)
    expected_keywords = normalize_list_field(expected_keywords)

    if results_df is None or len(results_df) == 0:
        return {
            f"precision_at_{k}": 0.0,
            f"recall_at_{k}": 0.0,
            f"mrr_at_{k}": 0.0,
            "ticker_hit": 0.0,
            "topic_hit": 0.0,
            "keyword_hit": 0.0,
            "relevant_count": 0,
        }

    top_df = results_df.head(k).copy()

    relevance = []
    ticker_hits = []
    topic_hits = []
    keyword_hits = []

    for _, row in top_df.iterrows():
        rel, th, toh, kh = row_is_relevant(row, expected_tickers, expected_topics, expected_keywords)
        relevance.append(rel)
        ticker_hits.append(th)
        topic_hits.append(toh)
        keyword_hits.append(kh)

    relevant_count = int(sum(relevance))

    # Correct @K denominator: divide by k, not by returned rows.
    precision = relevant_count / k
    recall = 1.0 if relevant_count > 0 else 0.0

    mrr = 0.0
    for idx, rel in enumerate(relevance, start=1):
        if rel:
            mrr = 1.0 / idx
            break

    return {
        f"precision_at_{k}": precision,
        f"recall_at_{k}": recall,
        f"mrr_at_{k}": mrr,
        "ticker_hit": float(any(ticker_hits)),
        "topic_hit": float(any(topic_hits)),
        "keyword_hit": float(any(keyword_hits)),
        "relevant_count": relevant_count,
    }

Retrieval wrapper for evaluation

In [21]:
def run_retrieval_with_config(query, config, auto_filter=True):
    mode = config["mode"]

    detected_tickers = detect_tickers_from_query(query) if auto_filter else None
    tickers = detected_tickers if detected_tickers else None

    if mode == "dense_only":
        return dense_search(
            query,
            top_k=config.get("final_top_k", 10),
            tickers=tickers,
            expand=config.get("expand", True)
        )

    if mode == "bm25_only":
        return bm25_search(
            query,
            top_k=config.get("final_top_k", 10),
            tickers=tickers,
            expand=config.get("expand", True)
        )

    if mode == "hybrid_rrf":
        return hybrid_search(
            query,
            dense_top_k=config.get("dense_top_k", 20),
            bm25_top_k=config.get("bm25_top_k", 20),
            final_top_k=config.get("final_top_k", 10),
            rrf_k=config.get("rrf_k", 60),
            tickers=tickers,
            expand=config.get("expand", True),
            diversify=config.get("diversify", True),
        )

    raise ValueError(f"Unknown retrieval mode: {mode}")

Build config grid

In [22]:
configs = []

# Evaluation should standardize final_top_k=10.
# We can still inspect top 5 via metrics, but retrieval config should return 10.
for expand in [False, True]:
    configs.append({
        "mode": "dense_only",
        "final_top_k": 10,
        "expand": expand,
    })

    configs.append({
        "mode": "bm25_only",
        "final_top_k": 10,
        "expand": expand,
    })

for dense_top_k in [10, 20, 30]:
    for bm25_top_k in [10, 20, 30]:
        for rrf_k in [30, 60, 90]:
            for expand in [False, True]:
                configs.append({
                    "mode": "hybrid_rrf",
                    "dense_top_k": dense_top_k,
                    "bm25_top_k": bm25_top_k,
                    "rrf_k": rrf_k,
                    "final_top_k": 10,
                    "expand": expand,
                    "diversify": True,
                })

print("Total configs:", len(configs))

Total configs: 58


Run automated retrieval evaluation

In [23]:
eval_records = []

for config_id, config in enumerate(tqdm(configs, desc="Evaluating configs")):
    query_metrics = []

    for query_id, query_row in eval_df.iterrows():
        results = run_retrieval_with_config(query_row["query"], config)

        metrics_at_5 = evaluate_single_query(
            results,
            query_row["expected_tickers"],
            query_row["expected_topics"],
            query_row["expected_keywords"],
            k=5
        )

        metrics_at_10 = evaluate_single_query(
            results,
            query_row["expected_tickers"],
            query_row["expected_topics"],
            query_row["expected_keywords"],
            k=10
        )

        record = {
            "config_id": config_id,
            "query_id": query_id,
            "query": query_row["query"],
            "mode": config["mode"],
            "dense_top_k": config.get("dense_top_k", np.nan),
            "bm25_top_k": config.get("bm25_top_k", np.nan),
            "rrf_k": config.get("rrf_k", np.nan),
            "final_top_k": config.get("final_top_k", np.nan),
            "expand": config.get("expand", False),
            "diversify": config.get("diversify", False),
            **metrics_at_5,
            **{k.replace("_10", "_10"): v for k, v in metrics_at_10.items()}
        }

        query_metrics.append(record)

    eval_records.extend(query_metrics)

retrieval_eval_df = pd.DataFrame(eval_records)

print("Evaluation rows:", len(retrieval_eval_df))
display(retrieval_eval_df.head())

Evaluating configs:   0%|          | 0/58 [00:00<?, ?it/s]

Evaluation rows: 1160


,config_id,query_id,query,mode,dense_top_k,bm25_top_k,rrf_k,final_top_k,expand,diversify,precision_at_5,recall_at_5,mrr_at_5,ticker_hit,topic_hit,keyword_hit,relevant_count,precision_at_10,recall_at_10,mrr_at_10
0,0,0,How does Nvidia discuss demand for AI infrastr...,dense_only,NaN,NaN,NaN,10,False,False,1.0,1.0,1.0,1.0,1.0,1.0,10,1.0,1.0,1.0
1,0,1,What does Nvidia say about export controls and...,dense_only,NaN,NaN,NaN,10,False,False,1.0,1.0,1.0,1.0,1.0,1.0,10,1.0,1.0,1.0
2,0,2,How does Microsoft discuss Azure and artificia...,dense_only,NaN,NaN,NaN,10,False,False,1.0,1.0,1.0,1.0,1.0,1.0,10,1.0,1.0,1.0
3,0,3,What does Microsoft say about cloud revenue an...,dense_only,NaN,NaN,NaN,10,False,False,1.0,1.0,1.0,1.0,1.0,1.0,10,1.0,1.0,1.0
4,0,4,What risks does Apple mention related to China?,dense_only,NaN,NaN,NaN,10,False,False,1.0,1.0,1.0,1.0,1.0,1.0,10,1.0,1.0,1.0


Aggregate evaluation results

In [24]:
metric_cols = [
    "precision_at_5",
    "recall_at_5",
    "mrr_at_5",
    "precision_at_10",
    "recall_at_10",
    "mrr_at_10",
    "ticker_hit",
    "topic_hit",
    "keyword_hit",
    "relevant_count",
]

config_summary_df = (
    retrieval_eval_df
    .groupby([
        "config_id", "mode", "dense_top_k", "bm25_top_k", "rrf_k",
        "final_top_k", "expand", "diversify"
    ], dropna=False)[metric_cols]
    .mean()
    .reset_index()
)

config_summary_df = config_summary_df.sort_values(
    ["mrr_at_10", "recall_at_10", "precision_at_10", "mrr_at_5", "ticker_hit"],
    ascending=False
).reset_index(drop=True)

display(config_summary_df.head(20))

best_config_id = int(config_summary_df.iloc[0]["config_id"])
best_config = configs[best_config_id]

print("Best config ID:", best_config_id)
print(json.dumps(best_config, indent=2))

,config_id,mode,dense_top_k,bm25_top_k,rrf_k,final_top_k,expand,diversify,precision_at_5,recall_at_5,mrr_at_5,precision_at_10,recall_at_10,mrr_at_10,ticker_hit,topic_hit,keyword_hit,relevant_count
0,47,hybrid_rrf,30.0,20.0,30.0,10,True,True,0.98,1.0,1.0,0.620,1.0,1.0,1.0,1.0,1.0,6.20
1,49,hybrid_rrf,30.0,20.0,60.0,10,True,True,0.98,1.0,1.0,0.620,1.0,1.0,1.0,1.0,1.0,6.20
2,51,hybrid_rrf,30.0,20.0,90.0,10,True,True,0.98,1.0,1.0,0.620,1.0,1.0,1.0,1.0,1.0,6.20
3,53,hybrid_rrf,30.0,30.0,30.0,10,True,True,0.97,1.0,1.0,0.620,1.0,1.0,1.0,1.0,1.0,6.20
4,55,hybrid_rrf,30.0,30.0,60.0,10,True,True,0.97,1.0,1.0,0.620,1.0,1.0,1.0,1.0,1.0,6.20
5,57,hybrid_rrf,30.0,30.0,90.0,10,True,True,0.97,1.0,1.0,0.620,1.0,1.0,1.0,1.0,1.0,6.20
6,35,hybrid_rrf,20.0,30.0,30.0,10,True,True,0.97,1.0,1.0,0.615,1.0,1.0,1.0,1.0,1.0,6.15
7,37,hybrid_rrf,20.0,30.0,60.0,10,True,True,0.97,1.0,1.0,0.615,1.0,1.0,1.0,1.0,1.0,6.15
8,39,hybrid_rrf,20.0,30.0,90.0,10,True,True,0.97,1.0,1.0,0.615,1.0,1.0,1.0,1.0,1.0,6.15
9,29,hybrid_rrf,20.0,20.0,30.0,10,True,True,0.97,1.0,1.0,0.610,1.0,1.0,1.0,1.0,1.0,6.10


Best config ID: 47
{
  "mode": "hybrid_rrf",
  "dense_top_k": 30,
  "bm25_top_k": 20,
  "rrf_k": 30,
  "final_top_k": 10,
  "expand": true,
  "diversify": true
}


Save evaluation results and best config

In [25]:
retrieval_eval_df.to_csv(EVAL_RESULTS_PATH, index=False)

best_config_payload = {
    "best_config_id": best_config_id,
    "best_config": best_config,
    "selection_metric_priority": [
        "mrr_at_10",
        "recall_at_10",
        "precision_at_10",
        "ticker_hit"
    ],
    "best_config_summary": config_summary_df.iloc[0].to_dict(),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "chunk_file": CHUNKS_PATH,
    "faiss_index_path": FAISS_INDEX_PATH,
    "bm25_index_path": BM25_INDEX_PATH,
    "metadata_path": CHUNK_METADATA_PATH,
}

with open(BEST_CONFIG_PATH, "w") as f:
    json.dump(best_config_payload, f, indent=2)

print("Saved eval results:", EVAL_RESULTS_PATH)
print("Saved best config:", BEST_CONFIG_PATH)

Saved eval results: /content/drive/MyDrive/finance-rag-analyst/data/evaluation/retrieval_eval_results.csv
Saved best config: /content/drive/MyDrive/finance-rag-analyst/data/evaluation/best_retrieval_config.json


Failure analysis

In [26]:
best_eval_df = retrieval_eval_df[retrieval_eval_df["config_id"] == best_config_id].copy()

worst_queries_df = best_eval_df.sort_values(
    ["mrr_at_10", "recall_at_10", "precision_at_10"],
    ascending=True
)[[
    "query_id",
    "query",
    "precision_at_10",
    "recall_at_10",
    "mrr_at_10",
    "ticker_hit",
    "topic_hit",
    "keyword_hit",
    "relevant_count"
]]

display(worst_queries_df.head(10))

,query_id,query,precision_at_10,recall_at_10,mrr_at_10,ticker_hit,topic_hit,keyword_hit,relevant_count
940,0,How does Nvidia discuss demand for AI infrastr...,0.5,1.0,1.0,1.0,1.0,1.0,5
941,1,What does Nvidia say about export controls and...,0.5,1.0,1.0,1.0,1.0,1.0,5
942,2,How does Microsoft discuss Azure and artificia...,0.5,1.0,1.0,1.0,1.0,1.0,5
943,3,What does Microsoft say about cloud revenue an...,0.5,1.0,1.0,1.0,1.0,1.0,5
944,4,What risks does Apple mention related to China?,0.5,1.0,1.0,1.0,1.0,1.0,5
945,5,How does Apple discuss services revenue?,0.5,1.0,1.0,1.0,1.0,1.0,5
946,6,What does Amazon say about AWS and cloud infra...,0.5,1.0,1.0,1.0,1.0,1.0,5
947,7,How does Amazon discuss capital expenditures a...,0.5,1.0,1.0,1.0,1.0,1.0,5
948,8,What does Google say about advertising revenue?,0.5,1.0,1.0,1.0,1.0,1.0,5
949,9,How does Google discuss cloud services and AI?,0.5,1.0,1.0,1.0,1.0,1.0,5


Retrieval diagnostics function

In [27]:
def retrieval_diagnostics(query):
    print("=" * 120)
    print("QUERY:", query)
    print("=" * 120)

    detected_tickers = detect_tickers_from_query(query)
    tickers = detected_tickers if detected_tickers else None

    print("Detected tickers:", tickers)

    dense_df = dense_search(query, top_k=5, tickers=tickers)
    bm25_df = bm25_search(query, top_k=5, tickers=tickers)
    hybrid_df = hybrid_search(query, final_top_k=5, tickers=tickers)

    print("\nDENSE TOP 5")
    display(dense_df[[
        "rank", "ticker", "form_type", "filing_date",
        "topic_labels", "retrieval_score", "text_preview"
    ]])

    print("\nBM25 TOP 5")
    display(bm25_df[[
        "rank", "ticker", "form_type", "filing_date",
        "topic_labels", "retrieval_score", "text_preview"
    ]])

    print("\nHYBRID TOP 5")
    display(hybrid_df[[
        "rank", "ticker", "form_type", "filing_date",
        "topic_labels", "retrieval_score", "source_modes", "text_preview"
    ]])

    dense_ids = set(dense_df["chunk_id"].tolist())
    bm25_ids = set(bm25_df["chunk_id"].tolist())
    hybrid_ids = set(hybrid_df["chunk_id"].tolist())

    print("\nOverlap diagnostics:")
    print("Dense ∩ BM25:", len(dense_ids & bm25_ids))
    print("Dense ∩ Hybrid:", len(dense_ids & hybrid_ids))
    print("BM25 ∩ Hybrid:", len(bm25_ids & hybrid_ids))

Demo queries

In [28]:
demo_queries = [
    "How is Nvidia describing AI infrastructure demand?",
    "What does Microsoft say about Azure and AI?",
    "What risks does Apple mention about China?",
    "How does Amazon discuss AWS capital expenditures?",
    "What does Google say about advertising revenue?",
]

for q in demo_queries:
    retrieval_diagnostics(q)

QUERY: How is Nvidia describing AI infrastructure demand?
Detected tickers: ['NVDA']

DENSE TOP 5


,rank,ticker,form_type,filing_date,topic_labels,retrieval_score,text_preview
0,1,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|climate_esg|c...,0.811456,"run AI models, product offerings, and services..."
1,2,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|data_centers|...,0.788526,"mation may be limited or incomplete, and our s..."
2,3,NVDA,10-K,2025-02-26,advertising|ai_strategy|business|cloud|custome...,0.776174,of the supercomputers on the global TOP500 lis...
3,4,NVDA,10-K,2024-02-21,advertising|ai_strategy|business|cloud|custome...,0.773778,of the supercomputers on the global TOP500 lis...
4,5,NVDA,10-K,2024-02-21,acquisitions|advertising|ai_strategy|business|...,0.769754,dous acceleration for applications. These plat...



BM25 TOP 5


,rank,ticker,form_type,filing_date,topic_labels,retrieval_score,text_preview
0,1,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|data_centers|...,35.364410,"mation may be limited or incomplete, and our s..."
1,2,NVDA,10-K,2025-02-26,advertising|ai_strategy|business|data_centers|...,33.365814,", and our statements should not be read to ind..."
2,3,NVDA,10-K,2024-02-21,advertising|ai_strategy|business|data_centers|...,33.348991,", and our statements should not be read to ind..."
3,4,NVDA,10-K,2026-02-25,ai_strategy|data_centers|privacy|semiconductor...,30.870051,or the AV market under the DRIVE Hyperion plat...
4,5,NVDA,10-K,2025-02-26,ai_strategy|cloud|data_centers|privacy|semicon...,29.764921,ying technology by using a variety of software...



HYBRID TOP 5


...


BM25 TOP 5


,rank,ticker,form_type,filing_date,topic_labels,retrieval_score,text_preview
0,1,GOOGL,10-K,2025-02-05,advertising|business|cloud|customer_demand|rev...,26.954330,or issuer to Google. We generate revenues by d...
1,2,GOOGL,10-K,2026-02-05,advertising|business|cloud|customer_demand|liq...,25.958391,"s, net 8 1, 154 400 Net cash used in financing..."
2,3,GOOGL,10-K,2024-01-31,advertising|ai_strategy|business|cloud|cyberse...,25.451629,vers from four years to six years and the esti...
3,4,GOOGL,10-K,2025-02-05,advertising|business|cloud|competition|custome...,22.603585,pect to continue hiring talented employees aro...
4,5,GOOGL,10-Q,2026-04-30,advertising|business|cloud|customer_demand|mac...,22.488179,e centralized certain AI-related research and ...



HYBRID TOP 5


,rank,ticker,form_type,filing_date,topic_labels,retrieval_score,source_modes,text_preview
0,1,GOOGL,10-K,2025-02-05,advertising|business|cloud|customer_demand|rev...,0.032787,bm25|dense,or issuer to Google. We generate revenues by d...
1,2,GOOGL,10-K,2024-01-31,advertising|ai_strategy|business|cloud|cyberse...,0.032002,bm25|dense,vers from four years to six years and the esti...
2,3,GOOGL,10-K,2025-02-05,advertising|business|cloud|competition|custome...,0.030777,bm25|dense,pect to continue hiring talented employees aro...
3,4,GOOGL,10-K,2026-02-05,advertising|ai_strategy|business|cloud|cyberse...,0.030550,bm25|dense,ers generally purchase advertising inventory t...
4,5,GOOGL,10-Q,2026-04-30,advertising|business|cloud|customer_demand|mac...,0.029877,bm25|dense,e centralized certain AI-related research and ...



Overlap diagnostics:
Dense ∩ BM25: 2
Dense ∩ Hybrid: 3
BM25 ∩ Hybrid: 4


Metadata-filtered retrieval demo

In [29]:
filtered_demo = hybrid_search(
    query="AI infrastructure and data center investment",
    tickers=["NVDA"],
    forms=["10-K"],
    final_top_k=5
)

display(filtered_demo[[
    "rank", "ticker", "form_type", "filing_date",
    "topic_labels", "retrieval_score", "source_modes", "text_preview", "source_url"
]])

,rank,ticker,form_type,filing_date,topic_labels,retrieval_score,source_modes,text_preview,source_url
0,1,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|climate_esg|c...,0.031514,bm25|dense,"run AI models, product offerings, and services...",https://www.sec.gov/Archives/edgar/data/104581...
1,2,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|data_centers|...,0.031099,bm25|dense,"mation may be limited or incomplete, and our s...",https://www.sec.gov/Archives/edgar/data/104581...
2,3,NVDA,10-K,2025-02-26,acquisitions|advertising|ai_strategy|business|...,0.029710,bm25|dense,ss our platforms strengthens our ecosystem and...,https://www.sec.gov/Archives/edgar/data/104581...
3,4,NVDA,10-K,2025-02-26,ai_strategy|business|china|customer_demand|dat...,0.029380,bm25|dense,". Recent Developments, Future Objectives and C...",https://www.sec.gov/Archives/edgar/data/104581...
4,5,NVDA,10-K,2024-02-21,ai_strategy|cloud|data_centers|semiconductors|...,0.029031,bm25|dense,AI systems for self-driving vehicles. Our unif...,https://www.sec.gov/Archives/edgar/data/104581...


Company filter safeguard tests

In [30]:
safeguard_tests = [
    ("What risks does Apple mention about China?", ["AAPL"]),
    ("What does Nvidia say about Blackwell and GPUs?", ["NVDA"]),
    ("What does Microsoft say about Azure and AI?", ["MSFT"]),
    ("How does Amazon discuss AWS capital expenditures?", ["AMZN"]),
    ("What does Google say about advertising revenue?", ["GOOGL"]),
]

for query, expected in safeguard_tests:
    detected = detect_tickers_from_query(query)
    results = hybrid_search(
        query,
        tickers=detected if detected else None,
        final_top_k=5
    )

    retrieved_tickers = sorted(results["ticker"].unique().tolist())

    print("\nQuery:", query)
    print("Detected:", detected)
    print("Retrieved:", retrieved_tickers)

    assert detected == expected, f"Detection failed for query: {query}"
    assert set(retrieved_tickers).issubset(set(expected)), f"Wrong ticker retrieved for query: {query}"

print("Company filter safeguards passed.")


Query: What risks does Apple mention about China?
Detected: ['AAPL']
Retrieved: ['AAPL']

Query: What does Nvidia say about Blackwell and GPUs?
Detected: ['NVDA']
Retrieved: ['NVDA']

Query: What does Microsoft say about Azure and AI?
Detected: ['MSFT']
Retrieved: ['MSFT']

Query: How does Amazon discuss AWS capital expenditures?
Detected: ['AMZN']
Retrieved: ['AMZN']

Query: What does Google say about advertising revenue?
Detected: ['GOOGL']
Retrieved: ['GOOGL']
Company filter safeguards passed.


Final validation

In [31]:
assert os.path.exists(EMBEDDINGS_PATH), "Embeddings file missing."
assert os.path.exists(FAISS_INDEX_PATH), "FAISS index missing."
assert os.path.exists(BM25_INDEX_PATH), "BM25 index missing."
assert os.path.exists(CHUNK_METADATA_PATH), "Chunk metadata missing."
assert os.path.exists(EVAL_RESULTS_PATH), "Eval results missing."
assert os.path.exists(BEST_CONFIG_PATH), "Best config file missing."

assert embeddings.shape[0] == len(chunks_df), "Embedding/chunk count mismatch."
assert faiss_index.ntotal == len(chunks_df), "FAISS/chunk count mismatch."
assert len(tokenized_corpus) == len(chunks_df), "BM25/chunk count mismatch."
assert len(retrieval_eval_df) > 0, "No evaluation results."
assert best_config["mode"] in ["dense_only", "bm25_only", "hybrid_rrf"], "Invalid best mode."

print("Notebook 3 completed successfully.")
print("Artifacts ready for Notebook 4:")
print("FAISS:", FAISS_INDEX_PATH)
print("BM25:", BM25_INDEX_PATH)
print("Metadata:", CHUNK_METADATA_PATH)
print("Best config:", BEST_CONFIG_PATH)

Notebook 3 completed successfully.
Artifacts ready for Notebook 4:
FAISS: /content/drive/MyDrive/finance-rag-analyst/data/indexes/faiss_bge_small.index
BM25: /content/drive/MyDrive/finance-rag-analyst/data/indexes/bm25_index.pkl
Metadata: /content/drive/MyDrive/finance-rag-analyst/data/indexes/chunk_metadata.parquet
Best config: /content/drive/MyDrive/finance-rag-analyst/data/evaluation/best_retrieval_config.json
